# MCI World Model v4.6.0 — Quickstart

**3 分钟上手因果世界模型**

> MCI World Model 是纯 NumPy 实现的因果推理引擎，零 GPU 依赖。

## 1. 安装与环境

```bash
pip install -e .    # 开发模式安装
# 完整版（含可视化）: pip install -e ".[full]"
```

In [ ]:
import numpy as np
import pandas as pd
from mci_world_model import CausalDataFrame, CausalGraphResult

np.random.seed(42)
print("MCI World Model 导入成功")

## 2. 因果发现 (Causal Discovery)

从观测数据中自动发现变量间的因果关系。支持 6 种算法。

In [ ]:
# 生成合成因果数据: X -> Y (X 导致 Y)
n = 200
X = np.random.randn(n)
Y = 0.8 * X + 0.3 * np.random.randn(n)
Z = 0.5 * X + 0.4 * np.random.randn(n)
data = np.column_stack([X, Y, Z])

cdf = CausalDataFrame(data)
cdf

In [ ]:
# PC 算法 (Peter-Clark)
g_pc = cdf.causal.discover(method="pc", alpha=0.05)
print(g_pc.summary())
# g_pc.show()  # 可视化 (需要 matplotlib+networkx)

In [ ]:
# 对比不同算法
algorithms = {
    "PC (条件独立)": cdf.causal.discover(method="pc"),
    "FCI (含隐混淆)": cdf.causal.discover(method="fci"),
    "NOTEARS (可微分)": cdf.causal.discover(method="notears"),
    "CAM (非线性加性)": cdf.causal.discover(method="cam"),
    "CAM+GOLEM (混合)": cdf.causal.discover(method="camgolem"),
}

for name, g in algorithms.items():
    print(f"{name}: {len(g.edges)} edges, conf={g.confidence:.3f}")

## 3. 干预推理 (Do-Calculus)

"如果我改变 X，Y 会如何变化？" — Pearl 因果层次第二层。

In [ ]:
from mci_world_model import DoCalculus, CausalGraph

cg = CausalGraph()
cg.add_node("X"); cg.add_node("M"); cg.add_node("Y")
cg.add_edge("X", "M"); cg.add_edge("M", "Y"); cg.add_edge("X", "Y")

dc = DoCalculus(cg)
result = dc.query(outcome="Y", intervention={"X": 1.0})
print(f"do(X=1.0) 后 P(Y): 可识别={result.identifiable}")

## 4. 反事实推理 (Counterfactual)

"如果当初做了不同的选择，结果会怎样？" — Pearl 因果层次第三层。

In [ ]:
from mci_world_model import CounterfactualEngine, StructuralEquationModel

sem = StructuralEquationModel()
sem.add_node("treatment", expr="U_T")
sem.add_node("outcome", expr="2*treatment + U_Y")

cf = CounterfactualEngine(sem)
result = cf.query(
    evidence={"treatment": 1, "outcome": 3},
    intervention={"treatment": 0},
    target="outcome"
)
print(f"反事实: 若 treatment=0, outcome={result.value:.2f}")

## 5. 时间序列因果 (Granger)

"过去的 X 是否包含预测 Y 的信息？"

In [ ]:
from mci_world_model import GrangerCausality

T = 200
X = np.cumsum(np.random.randn(T))
Y = 0.5 * np.roll(X, 1) + np.random.randn(T) * 0.3
Y[0] = np.random.randn()

gc = GrangerCausality(max_lag=5, significance=0.05)
report = gc.test(X, Y)
print(report.summary())

## 6. 端到端因果推理流水线

从数据 -> 因果发现 -> 干预推理 -> 反事实

In [ ]:
def causal_pipeline(data, intervention_var=None, intervention_val=None):
    from mci_world_model import DoCalculus, CausalGraph
    
    cdf = CausalDataFrame(data)
    graph = cdf.causal.discover(method="camgolem")
    print(f"[Step 1] 发现 {len(graph.edges)} 条因果边")
    
    cg = CausalGraph()
    for node in graph.nodes:
        cg.add_node(node)
    for src, dst in graph.edges:
        cg.add_edge(src, dst)
    
    if intervention_var:
        dc = DoCalculus(cg)
        result = dc.query(outcome=graph.nodes[-1],
                          intervention={intervention_var: intervention_val})
        print(f"[Step 2] do({intervention_var}={intervention_val}) -> 可识别={result.identifiable}")
    
    return graph

g = causal_pipeline(data, intervention_var="X0", intervention_val=1.0)

## 下一步

- 📖 完整 API 文档: `README.md`
- 🧪 运行测试: `PYTHONPATH=src pytest tests/ -q`
- 📊 基准测试: `benchmarks/` 目录
- 🏥 行业 SDK: `sdk/_medical_causal_sdk.py`